In [1]:
cd /root/hardware_rounding_error_predictor
python3 build_catalog.py --seq-lens 100 250 1000 4000 --no-verify 2>&1 | tee /root/catalog_build.log

SKU=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | tr -d ' ')
cat > /root/demo.sh << EOF
#!/bin/bash
cd /root/hardware_rounding_error_predictor
export HF_HOME=/root/hf_cache MUFU_CACHE_DIR=/root/mufu_cache
for SEQ in 100 250 1000 4000; do
    echo "================ seq=\$SEQ ================"
    python3 ffn_chain_test.py all \$SEQ
    echo "---- cuBLAS mode ----"
    EMULATOR_TARGET=cublas python3 ffn_chain_test.py compare \$SEQ
done
EOF
bash /root/demo.sh 2>&1 | tee /root/ffn_demo_${SKU}.log

M grid restricted to 4 specified seq lengths: [100, 250, 1000, 4000]

=== down_proj (N=2560, K=9728) ===
  [ 1/ 4] M=  100 N= 2560 K= 9728 /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  recipe=split_k_workspace_outtype    split_k=3  verify=skipped               kernel=ampere_bf16_s16816gemm_bf16_64x128_ldg8_f2f_stages_64x3_tn
  [ 2/ 4] M=  250 N= 2560 K= 9728   recipe=split_k_workspace_outtype    split_k=3  verify=skipped               kernel=cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_128x128_32x5_tn_align8
  [ 3/ 4] M= 1000 N= 2560 K= 9728   recipe=split_k_cutlass_bf16_out     split_k=3  verify=skipped               kernel=cutlass_80_tensorop_bf16_s16816gemm_relu_bf16_128x256_32x3_tn_align8
  [ 4/ 4] M= 4000 N= 2560 K= 9728   recipe=split_k_cutlass_bf16_out     split_k=3 